# ConvKAN-Derm v7: Heterogeneous Deep Ensemble
## EfficientNetV2-L + ConvNeXt-Base + Swin-Base @ 384x384

### v6 Baseline → v7 Upgrade Strategy
v6 bottleneck: **homogeneous ensemble** (all EfficientNet family) limits diversity.
v7 fix: **three distinct architectural paradigms** (local CNN + modern CNN + ViT).

| Component | v6 (Frozen Baseline) | v7 (This Notebook) |
|---|---|---|
| Backbone A | EffNetV2-M 54M (ImageNet-1k) | **EffNetV2-L 119M (ImageNet-22k)** |
| Backbone B | EffNetV2-S 21M seed=123 | **ConvNeXt-Base 89M (ImageNet-22k)** |
| Backbone C | EffNetV2-S 21M seed=456 | **Swin-Base 88M (ImageNet-22k)** |
| Resolution | 300×300 | **384×384** |
| TTA Passes | 30 (3×10) | **45 (3×15)** |
| Scheduler | OneCycleLR | **CosineAnnealingWarmRestarts (T0=10, Tm=2)** |
| Weight Decay | 5e-4 | **0.05 (aggressive regularization)** |
| Label Smoothing | 0.05 | **0.10** |
| Gradient Accum. | — | **4 steps (effective batch=64)** |
| KAN Input | 256-d (1 backbone) | **768-d (3× 256-d concatenated)** |

### Training Protocol
- Models A, B, C trained **independently** (no leakage between them)
- Each uses **4-stage gradual unfreezing** (head warmup → 25% → 50% → full)
- Soft-Vote Ensemble after all three are trained
- **HAM10000**: Master dataset (target >95%)
- **ISIC 2019**: Zero-shot cross-validation (target >90-92%)

> **v6 is completely frozen as baseline. Do not overwrite any v6 file.**


In [7]:
# 1. Clone the repository locally
!git clone https://github.com/Blealtan/efficient-kan.git

# 2. Install the local package using a valid relative path specifier
!pip install -q ./efficient-kan/
!pip install -q timm kagglehub torchvision pandas scikit-learn pillow matplotlib seaborn

import timm
print(f"timm version: {timm.__version__}")

Cloning into 'efficient-kan'...
fatal: unable to access 'https://github.com/Blealtan/efficient-kan.git/': Could not resolve host: github.com
ERROR: Invalid requirement: './efficient-kan/': Expected package name at the start of dependency specifier
    ./efficient-kan/
    ^
Hint: It looks like a path. File './efficient-kan/' does not exist.
timm version: 1.0.26


In [3]:
import os, glob, warnings, copy, math
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Primary Device: {device}")

if device.type == 'cuda':
    gpu_count = torch.cuda.device_count()
    print(f"Available GPUs: {gpu_count}")
    for i in range(gpu_count):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    # VRAM indicator for primary tracking
    print(f"Primary VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Flag to enable DataParallel if using T4 x2
    USE_DATA_PARALLEL = gpu_count > 1
else:
    USE_DATA_PARALLEL = False
    print("WARNING: Hardware accelerator not detected. Ensure GPU is enabled in Kaggle Session Options.")

Primary Device: cpu


In [4]:
import kagglehub
path = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')

image_path_dict = {
    os.path.splitext(os.path.basename(x))[0]: x
    for x in glob.glob(os.path.join(path, '*', '*.jpg'))
}

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
NUM_CLASSES  = len(CLASS_NAMES)
MEL_IDX      = CLASS_NAMES.index('mel')
NV_IDX       = CLASS_NAMES.index('nv')
label_map    = {v: i for i, v in enumerate(CLASS_NAMES)}

df = pd.read_csv(os.path.join(path, 'HAM10000_metadata.csv'))
df['label_idx'] = df['dx'].map(label_map)
df['path']      = df['image_id'].map(image_path_dict)
df = df.dropna(subset=['path']).reset_index(drop=True)
print(f'Total images: {len(df)}')
print(df['dx'].value_counts())


Total images: 10015
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


### 5. Dataset, Transforms & Class Weights

**v7 key upgrades vs v6:**
- `IMG_SIZE = 384` (up from 300) — forces models to resolve microscopic dermoscopic structures
- `RandAugment(num_ops=2, magnitude=12)` (up from 10)
- `ColorJitter` bounds slightly widened
- `label_smoothing = 0.10` in FocalLoss (up from 0.05)
- `weight_decay = 0.05` across all optimizers (up from 5e-4)


In [5]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import train_test_split

# ── v7 Hyperparameter Constants ──────────────────────────────────────────
IMG_SIZE    = 384         # upgraded from 300 in v6
BATCH_SIZE  = 16          # T4 RAM limit with 384x384 + large models
ACCUM_STEPS = 4           # effective batch = 16 * 4 = 64
NUM_WORKERS = 2

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

class HAM10000Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img   = Image.open(self.df.iloc[idx]['path']).convert('RGB')
        label = torch.tensor(int(self.df.iloc[idx]['label_idx']), dtype=torch.long)
        if self.transform: img = self.transform(img)
        return img, label

train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.55, 1.0)),    # slightly wider crop range
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(45),
    T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.45, hue=0.15),
    T.RandAugment(num_ops=2, magnitude=12),              # magnitude 12 vs 10 in v6
    T.RandomApply([T.GaussianBlur(3)], p=0.3),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.30, scale=(0.02, 0.25)),
])

val_transform = T.Compose([
    T.Resize(int(IMG_SIZE * 256 / 224)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_idx']
)

# Capped Focal Loss weights (4:1 max ratio, same formula as v4/v5/v6)
class_counts   = train_df['label_idx'].value_counts().sort_index().values.astype(float)
raw_weights    = 1.0 / (class_counts + 1e-6)
raw_weights    = raw_weights / raw_weights.min()
capped_weights = np.clip(raw_weights, 1.0, 4.0)
capped_weights = capped_weights / capped_weights.sum() * NUM_CLASSES

print('v7 Class weights (Capped Focal Loss, max 4:1):')
for n, w, c in zip(CLASS_NAMES, capped_weights, class_counts.astype(int)):
    print(f'  {n:6s}: weight={w:.3f}  n={c}')

val_loader = DataLoader(
    HAM10000Dataset(val_df, val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

def make_train_loader(df_split, seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    return DataLoader(
        HAM10000Dataset(df_split, train_transform),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True
    )

print(f'\nIMG_SIZE={IMG_SIZE} | BATCH_SIZE={BATCH_SIZE} | ACCUM_STEPS={ACCUM_STEPS}')
print(f'Effective batch size: {BATCH_SIZE * ACCUM_STEPS}')
print(f'Val batches: {len(val_loader)}')


NameError: name 'Dataset' is not defined

### 6. v7 Heterogeneous Model Architectures

Three fundamentally different inductive biases:

| ID | Backbone | Params | Paradigm | Key Strength |
|---|---|---|---|---|
| A | `tf_efficientnetv2_l.in21k` | ~119M | Local CNN (compound scaled) | Fine-grained local textures |
| B | `convnext_base.fb_in22k` | ~89M | Modernised CNN (ViT design principles) | Global-local hybrid features |
| C | `swin_base_patch4_window12_384.ms_in22k` | ~88M | Shifted-Window ViT | Multi-scale global context |

All share a unified MLP head: `feat_dim → BN → Dropout → 512 → BN → ReLU → 256 → BN → ReLU → 7`
The 256-dim layer is the **KAN bottleneck** (concatenated → 768-d for ensemble KAN head).


In [ ]:
# ── Backbone configuration registry ──────────────────────────────────────
BACKBONE_CFG = {
    'A': dict(
        timm_name='tf_efficientnetv2_l.in21k',
        feat_dim=1280,
        ckpt='best_v7_A_effv2l.pth',
        label='EfficientNetV2-L (ImageNet-22k)'
    ),
    'B': dict(
        timm_name='convnext_base.fb_in22k',
        feat_dim=1024,
        ckpt='best_v7_B_convnext.pth',
        label='ConvNeXt-Base (ImageNet-22k)'
    ),
    'C': dict(
        timm_name='swin_base_patch4_window12_384.ms_in22k',
        feat_dim=1024,
        ckpt='best_v7_C_swinb.pth',
        label='Swin-Base 384 (ImageNet-22k)'
    ),
}

class DermModel_v7(nn.Module):
    def __init__(self, backbone_key, num_classes=7, dropout=0.4):
        super().__init__()
        cfg = BACKBONE_CFG[backbone_key]
        self.backbone_key = backbone_key
        self.label = cfg['label']
        
        # Core Backbone (pooling features to vectors)
        self.backbone = timm.create_model(
            cfg['timm_name'], 
            pretrained=True, 
            num_classes=0, 
            global_pool='avg'
        )
        
        # Unified projection MLP head down to the 256-d KAN bottleneck
        self.head = nn.Sequential(
            nn.BatchNorm1d(cfg['feat_dim']),
            nn.Dropout(p=dropout),
            nn.Linear(cfg['feat_dim'], 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, 256),  # <--- This is the KAN bottleneck layer
            nn.BatchNorm1d(256),
            nn.ReLU()
        )
        
        # Final classification projection layer
        self.classifier = nn.Linear(256, num_classes)

    def get_features(self, x):
        """Extracts the 256-d features for KAN head concatenation."""
        x = self.backbone(x)
        x = self.head(x)
        return x

    def forward(self, x):
        features = self.get_features(x)
        return self.classifier(features)

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_all(self):
        for param in self.backbone.parameters():
            param.requires_grad = True

    def unfreeze_last_fraction(self, fraction):
        """Unfreezes the top fraction of layers for gradual unfreezing stages."""
        self.unfreeze_all()
        # Estimate layers based on immediate children named modules
        modules = list(self.backbone.children())
        num_modules = len(modules)
        freeze_until = int(num_modules * (1 - fraction))
        
        for i, module in enumerate(modules[:freeze_until]):
            for param in module.parameters():
                param.requires_grad = False

class DermModel_v7(nn.Module):
    def __init__(self, backbone_key, num_classes=NUM_CLASSES, dropout=0.4):
        super().__init__()
        cfg = BACKBONE_CFG[backbone_key]
        self.backbone_key = backbone_key
        self.label        = cfg['label']
        # timm backbone — returns pooled feature vector (num_classes=0)
        self.backbone = timm.create_model(
            cfg['timm_name'], pretrained=True,
            num_classes=0, global_pool='avg'
        )
        feat_dim = cfg['feat_dim']
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def get_features(self, x):
        feat = self.backbone(x)               # [B, feat_dim]
        for layer in list(self.head.children())[:-1]:
            feat = layer(feat)
        return feat                            # [B, 256]

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False

    def unfreeze_last_fraction(self, fraction):
        params = list(self.backbone.parameters())
        n_freeze = int(len(params) * (1.0 - fraction))
        for i, p in enumerate(params):
            p.requires_grad = (i >= n_freeze)
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'  [{self.backbone_key}] Last {fraction*100:.0f}% unfrozen | Trainable: {trainable:,}')

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad = True

# ── Instantiate all three models ──────────────────────────────────────────
print('Building Model A — EfficientNetV2-L ...')
model_A = DermModel_v7('A').to(device)
print(f'  Total params: {sum(p.numel() for p in model_A.parameters()):,}')

print('Building Model B — ConvNeXt-Base ...')
model_B = DermModel_v7('B').to(device)
print(f'  Total params: {sum(p.numel() for p in model_B.parameters()):,}')

print('Building Model C — Swin-Base 384 ...')
model_C = DermModel_v7('C').to(device)
print(f'  Total params: {sum(p.numel() for p in model_C.parameters()):,}')

print('\nAll three v7 backbones loaded with ImageNet-22k weights.')


In [ ]:
class FocalLoss_v7(nn.Module):
    def __init__(self, alpha=None, gamma=2, label_smoothing=0.10):
        super().__init__()
        self.alpha           = alpha
        self.gamma           = gamma
        self.label_smoothing = label_smoothing  # 0.10 vs 0.05 in v6

    def forward(self, inputs, targets):
        ce  = F.cross_entropy(
            inputs, targets,
            weight=self.alpha,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )
        pt  = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

class_weights_tensor = torch.tensor(capped_weights, dtype=torch.float).to(device)
criterion = FocalLoss_v7(alpha=class_weights_tensor, gamma=2, label_smoothing=0.10)
print('Loss: Capped Focal Loss (max 4:1) + label_smoothing=0.10  [upgraded from 0.05 in v6]')


In [ ]:
# ── MixUp ─────────────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(x.size(0)).to(x.device)
    mixed = lam * x + (1 - lam) * x[idx]
    return mixed, y, y[idx], lam

# ── CutMix ────────────────────────────────────────────────────────────────
def cutmix_data(x, y, alpha=1.0):
    lam     = np.random.beta(alpha, alpha)
    idx     = torch.randperm(x.size(0)).to(x.device)
    H, W    = x.size(2), x.size(3)
    cut_r   = np.sqrt(1 - lam)
    ch, cw  = int(H * cut_r), int(W * cut_r)
    cy, cx  = np.random.randint(H), np.random.randint(W)
    y1, y2  = max(0, cy - ch // 2), min(H, cy + ch // 2)
    x1, x2  = max(0, cx - cw // 2), min(W, cx + cw // 2)
    mixed   = x.clone()
    mixed[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam     = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    return mixed, y, y[idx], lam

def mixed_criterion(crit, pred, ya, yb, lam):
    return lam * crit(pred, ya) + (1 - lam) * crit(pred, yb)

# ── Training loop (gradient accumulation) ────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scaler,
                    use_aug=True, alpha=0.4, accum_steps=ACCUM_STEPS):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for step, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        r = np.random.random()
        if use_aug and r < 0.30:
            images, ya, yb, lam = cutmix_data(images, labels, alpha)
            with autocast('cuda'):
                loss = mixed_criterion(criterion, model(images), ya, yb, lam)
        elif use_aug and r < 0.60:
            images, ya, yb, lam = mixup_data(images, labels, alpha)
            with autocast('cuda'):
                loss = mixed_criterion(criterion, model(images), ya, yb, lam)
        else:
            with autocast('cuda'):
                loss = criterion(model(images), labels)
        loss = loss / accum_steps
        scaler.scale(loss).backward()
        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        total_loss += loss.item() * accum_steps
    return total_loss / len(loader)

def get_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            preds = model(imgs.to(device)).argmax(dim=1)
            y_true.extend(lbls.numpy())
            y_pred.extend(preds.cpu().numpy())
    return np.array(y_true), np.array(y_pred)

# ── run_stage — uses CosineAnnealingWarmRestarts (epoch-level) ────────────
def run_stage(model, train_loader, val_loader, optimizer, criterion,
              scheduler, num_epochs, stage_name, checkpoint_path,
              patience=12, use_aug=True, alpha=0.4, accum_steps=ACCUM_STEPS):
    scaler  = GradScaler('cuda')
    best_f1, pat_cnt = 0.0, 0
    history = {'loss': [], 'val_f1': [], 'val_acc': []}
    for epoch in range(num_epochs):
        loss = train_one_epoch(
            model, train_loader, optimizer, criterion,
            scaler, use_aug=use_aug, alpha=alpha, accum_steps=accum_steps
        )
        scheduler.step()           # CAWR: epoch-level step
        y_t, y_p = get_predictions(model, val_loader)
        f1  = f1_score(y_t, y_p, average='macro', zero_division=0)
        acc = np.mean(y_t == y_p)
        history['loss'].append(loss)
        history['val_f1'].append(f1)
        history['val_acc'].append(acc)
        flag = ' <- BEST' if f1 > best_f1 else ''
        print(f'[{stage_name}] Ep {epoch+1:02d}/{num_epochs} | '
              f'Loss {loss:.4f} | Acc {acc*100:.1f}% | F1 {f1:.4f}{flag}')
        if f1 > best_f1:
            best_f1, pat_cnt = f1, 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            pat_cnt += 1
            if pat_cnt >= patience:
                print(f'  Early stop at epoch {epoch+1}.')
                break
    print(f'[{stage_name}] Best Macro F1: {best_f1:.4f} -> {checkpoint_path}')
    return history

# ── Set up backbone unfreezing shortcut ──────────────────────────────────
def make_cawr_optimizer(model, head_lr=1e-4, backbone_lr=1e-5, wd=0.05):
    backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
    head_params     = list(model.head.parameters())
    groups = []
    if backbone_params:
        groups.append({'params': backbone_params, 'lr': backbone_lr})
    if head_params:
        groups.append({'params': head_params, 'lr': head_lr})
    opt = torch.optim.AdamW(groups, weight_decay=wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=10, T_mult=2, eta_min=1e-6
    )
    return opt, sch

print('Training utilities ready.')
print(f'Scheduler: CosineAnnealingWarmRestarts(T_0=10, T_mult=2)')
print(f'Gradient accumulation: {ACCUM_STEPS} steps (effective batch = {BATCH_SIZE*ACCUM_STEPS})')


### 9. Model A (EfficientNetV2-L) — 4-Stage Gradual Unfreezing

**Rationale**: EfficientNetV2-L at 384×384 is the highest-capacity backbone we train.
4-stage gradual unfreezing prevents catastrophic forgetting of ImageNet-22k features.

| Stage | Backbone State | Head LR | Backbone LR | Epochs | Aug |
|---|---|---|---|---|---|
| S1 | Fully Frozen | 1e-4 | — | 20 | OFF |
| S2 | Last 25% unfrozen | 1e-4 | 1e-5 | 20 | ON |
| S3 | Last 50% unfrozen | 1e-4 | 1e-5 | 20 | ON |
| S4 | Full fine-tune | 1e-4 | 1e-5 | 40 | ON |


In [ ]:
torch.manual_seed(42); np.random.seed(42)
train_loader_A = make_train_loader(train_df, seed=42)

model_A.freeze_backbone()
opt_A1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_A.parameters()),
    lr=1e-4, weight_decay=0.05
)
sch_A1 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt_A1, T_0=10, T_mult=2, eta_min=1e-6
)

print('=== Model A — Stage 1: Head warmup | EfficientNetV2-L FROZEN ===')
hist_A1 = run_stage(
    model_A, train_loader_A, val_loader,
    opt_A1, criterion, sch_A1,
    num_epochs=20, stage_name='A-S1-Warmup',
    checkpoint_path=BACKBONE_CFG['A']['ckpt'],
    patience=20, use_aug=False
)


In [ ]:
model_A.load_state_dict(torch.load(BACKBONE_CFG['A']['ckpt'], weights_only=True))
model_A.unfreeze_last_fraction(0.25)
opt_A2, sch_A2 = make_cawr_optimizer(model_A, head_lr=1e-4, backbone_lr=1e-5)

print('=== Model A — Stage 2: Last 25% unfrozen | CutMix+MixUp ON ===')
hist_A2 = run_stage(
    model_A, train_loader_A, val_loader,
    opt_A2, criterion, sch_A2,
    num_epochs=20, stage_name='A-S2-U25',
    checkpoint_path=BACKBONE_CFG['A']['ckpt'],
    patience=15, use_aug=True
)


In [ ]:
model_A.load_state_dict(torch.load(BACKBONE_CFG['A']['ckpt'], weights_only=True))
model_A.unfreeze_last_fraction(0.50)
opt_A3, sch_A3 = make_cawr_optimizer(model_A, head_lr=1e-4, backbone_lr=1e-5)

print('=== Model A — Stage 3: Last 50% unfrozen ===')
hist_A3 = run_stage(
    model_A, train_loader_A, val_loader,
    opt_A3, criterion, sch_A3,
    num_epochs=20, stage_name='A-S3-U50',
    checkpoint_path=BACKBONE_CFG['A']['ckpt'],
    patience=15, use_aug=True
)


In [ ]:
model_A.load_state_dict(torch.load(BACKBONE_CFG['A']['ckpt'], weights_only=True))
model_A.unfreeze_all()
opt_A4, sch_A4 = make_cawr_optimizer(model_A, head_lr=1e-4, backbone_lr=1e-5)

print('=== Model A — Stage 4: FULL fine-tune | 40 epochs ===')
print('Weight Decay = 0.05 | CAWR T0=10 T_mult=2 | Gradient Accum =', ACCUM_STEPS)
hist_A4 = run_stage(
    model_A, train_loader_A, val_loader,
    opt_A4, criterion, sch_A4,
    num_epochs=40, stage_name='A-S4-Full',
    checkpoint_path=BACKBONE_CFG['A']['ckpt'],
    patience=12, use_aug=True
)


In [ ]:
def predict_tta_single(model, loader, n=15):
    tta_tf = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomRotation(15),
        T.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0))
    ])
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs  = imgs.to(device)
            probs = sum(
                F.softmax(model(torch.stack([tta_tf(i) for i in imgs])), dim=1)
                for _ in range(n)
            )
            y_true.extend(lbls.numpy())
            y_pred.extend(probs.argmax(dim=1).cpu().numpy())
    return np.array(y_true), np.array(y_pred)

model_A.load_state_dict(torch.load(BACKBONE_CFG['A']['ckpt'], weights_only=True))
print('Evaluating Model A (EfficientNetV2-L) with TTA-15...')
y_A, p_A = predict_tta_single(model_A, val_loader, n=15)
acc_A = np.mean(y_A == p_A)
f1_A  = f1_score(y_A, p_A, average='macro', zero_division=0)

print(f'\n[Model A TTA-15] Accuracy : {acc_A*100:.2f}%')
print(f'[Model A TTA-15] Macro F1 : {f1_A:.4f}')
print(f'[v6 Ens TTA-30]  Baseline : (see v6 notebook for frozen results)')


### 14. Model B (ConvNeXt-Base) — 4-Stage Training

ConvNeXt-Base operates with pure CNN inductive biases but modernised with:
- Depthwise conv with large kernels (7×7 receptive field per block)
- Inverted bottleneck design (borrowed from ViTs)
- LayerNorm instead of BatchNorm

This gives **Vision-Transformer-level accuracy** while retaining CNN's translation equivariance —
critical for dermoscopy where lesion position is variable.


In [ ]:
torch.manual_seed(123); np.random.seed(123)
train_loader_B = make_train_loader(train_df, seed=123)

model_B.freeze_backbone()
opt_B1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_B.parameters()),
    lr=1e-4, weight_decay=0.05
)
sch_B1 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt_B1, T_0=10, T_mult=2, eta_min=1e-6
)
print('=== Model B — Stage 1: ConvNeXt-Base head warmup ===')
hist_B1 = run_stage(
    model_B, train_loader_B, val_loader,
    opt_B1, criterion, sch_B1,
    num_epochs=20, stage_name='B-S1-Warmup',
    checkpoint_path=BACKBONE_CFG['B']['ckpt'],
    patience=20, use_aug=False
)


In [ ]:
model_B.load_state_dict(torch.load(BACKBONE_CFG['B']['ckpt'], weights_only=True))
model_B.unfreeze_last_fraction(0.25)
opt_B2, sch_B2 = make_cawr_optimizer(model_B, head_lr=1e-4, backbone_lr=1e-5)
print('=== Model B — Stage 2: Last 25% unfrozen ===')
hist_B2 = run_stage(
    model_B, train_loader_B, val_loader,
    opt_B2, criterion, sch_B2,
    num_epochs=20, stage_name='B-S2-U25',
    checkpoint_path=BACKBONE_CFG['B']['ckpt'],
    patience=15, use_aug=True
)


In [ ]:
model_B.load_state_dict(torch.load(BACKBONE_CFG['B']['ckpt'], weights_only=True))
model_B.unfreeze_last_fraction(0.50)
opt_B3, sch_B3 = make_cawr_optimizer(model_B, head_lr=1e-4, backbone_lr=1e-5)
print('=== Model B — Stage 3: Last 50% unfrozen ===')
hist_B3 = run_stage(
    model_B, train_loader_B, val_loader,
    opt_B3, criterion, sch_B3,
    num_epochs=20, stage_name='B-S3-U50',
    checkpoint_path=BACKBONE_CFG['B']['ckpt'],
    patience=15, use_aug=True
)


In [ ]:
model_B.load_state_dict(torch.load(BACKBONE_CFG['B']['ckpt'], weights_only=True))
model_B.unfreeze_all()
opt_B4, sch_B4 = make_cawr_optimizer(model_B, head_lr=1e-4, backbone_lr=1e-5)
print('=== Model B — Stage 4: Full fine-tune | 40 epochs ===')
hist_B4 = run_stage(
    model_B, train_loader_B, val_loader,
    opt_B4, criterion, sch_B4,
    num_epochs=40, stage_name='B-S4-Full',
    checkpoint_path=BACKBONE_CFG['B']['ckpt'],
    patience=12, use_aug=True
)


In [ ]:
model_B.load_state_dict(torch.load(BACKBONE_CFG['B']['ckpt'], weights_only=True))
print('Evaluating Model B (ConvNeXt-Base) with TTA-15...')
y_B, p_B = predict_tta_single(model_B, val_loader, n=15)
acc_B = np.mean(y_B == p_B)
f1_B  = f1_score(y_B, p_B, average='macro', zero_division=0)
print(f'[Model B TTA-15] Accuracy : {acc_B*100:.2f}%')
print(f'[Model B TTA-15] Macro F1 : {f1_B:.4f}')


### 15. Model C (Swin-Transformer-Base @ 384×384) — 4-Stage Training

Swin-B operates via **Shifted Window Self-Attention**:
- Partitions image into non-overlapping windows (12×12 at 384×384 = 1024 tokens)
- Self-attention computed *within* windows; shifted windows capture cross-window context
- Hierarchical feature maps at 4 scales (like CNN feature pyramids)

**Why critical for melanoma**: global asymmetry features + fine border irregularities
are captured simultaneously via multi-scale self-attention — impossible for pure local CNNs.


In [ ]:
torch.manual_seed(456); np.random.seed(456)
train_loader_C = make_train_loader(train_df, seed=456)

model_C.freeze_backbone()
opt_C1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_C.parameters()),
    lr=1e-4, weight_decay=0.05
)
sch_C1 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt_C1, T_0=10, T_mult=2, eta_min=1e-6
)
print('=== Model C — Stage 1: Swin-Base head warmup ===')
hist_C1 = run_stage(
    model_C, train_loader_C, val_loader,
    opt_C1, criterion, sch_C1,
    num_epochs=20, stage_name='C-S1-Warmup',
    checkpoint_path=BACKBONE_CFG['C']['ckpt'],
    patience=20, use_aug=False
)


In [ ]:
model_C.load_state_dict(torch.load(BACKBONE_CFG['C']['ckpt'], weights_only=True))
model_C.unfreeze_last_fraction(0.25)
opt_C2, sch_C2 = make_cawr_optimizer(model_C, head_lr=1e-4, backbone_lr=1e-5)
print('=== Model C — Stage 2: Last 25% unfrozen ===')
hist_C2 = run_stage(
    model_C, train_loader_C, val_loader,
    opt_C2, criterion, sch_C2,
    num_epochs=20, stage_name='C-S2-U25',
    checkpoint_path=BACKBONE_CFG['C']['ckpt'],
    patience=15, use_aug=True
)


In [ ]:
model_C.load_state_dict(torch.load(BACKBONE_CFG['C']['ckpt'], weights_only=True))
model_C.unfreeze_last_fraction(0.50)
opt_C3, sch_C3 = make_cawr_optimizer(model_C, head_lr=1e-4, backbone_lr=1e-5)
print('=== Model C — Stage 3: Last 50% unfrozen ===')
hist_C3 = run_stage(
    model_C, train_loader_C, val_loader,
    opt_C3, criterion, sch_C3,
    num_epochs=20, stage_name='C-S3-U50',
    checkpoint_path=BACKBONE_CFG['C']['ckpt'],
    patience=15, use_aug=True
)


In [ ]:
model_C.load_state_dict(torch.load(BACKBONE_CFG['C']['ckpt'], weights_only=True))
model_C.unfreeze_all()
opt_C4, sch_C4 = make_cawr_optimizer(model_C, head_lr=1e-4, backbone_lr=1e-5)
print('=== Model C — Stage 4: Full fine-tune | 40 epochs ===')
hist_C4 = run_stage(
    model_C, train_loader_C, val_loader,
    opt_C4, criterion, sch_C4,
    num_epochs=40, stage_name='C-S4-Full',
    checkpoint_path=BACKBONE_CFG['C']['ckpt'],
    patience=12, use_aug=True
)


In [ ]:
model_C.load_state_dict(torch.load(BACKBONE_CFG['C']['ckpt'], weights_only=True))
print('Evaluating Model C (Swin-Base 384) with TTA-15...')
y_C, p_C = predict_tta_single(model_C, val_loader, n=15)
acc_C = np.mean(y_C == p_C)
f1_C  = f1_score(y_C, p_C, average='macro', zero_division=0)
print(f'[Model C TTA-15] Accuracy : {acc_C*100:.2f}%')
print(f'[Model C TTA-15] Macro F1 : {f1_C:.4f}')
print('\n--- Single-Model Summary ---')
print(f'  A (EffNetV2-L):  Acc={acc_A*100:.2f}%  F1={f1_A:.4f}')
print(f'  B (ConvNeXt-B):  Acc={acc_B*100:.2f}%  F1={f1_B:.4f}')
print(f'  C (Swin-B 384):  Acc={acc_C*100:.2f}%  F1={f1_C:.4f}')


### 16. Heterogeneous 3-Model Ensemble — Soft-Vote + TTA-45 (3×15)

Each of the 3 models runs 15 TTA passes per image.
Total effective predictions per sample: **3 × 15 = 45 averaged softmax vectors**.

The ensemble gains power specifically because:
- **Model A** (EffNetV2-L): dominates on fine texture features (jagged borders)
- **Model B** (ConvNeXt): compensates on shape/structure features
- **Model C** (Swin-B): provides global lesion context (symmetry axis, colour blobs)

Their errors are **architecturally uncorrelated** — the soft-vote averages them out.


In [ ]:
def ensemble_predict_tta(models_list, loader, n_tta=15):
    tta_tf = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomRotation(15),
        T.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0))
    ])
    for m in models_list: m.eval()
    y_true, all_probs = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs  = imgs.to(device)
            probs = torch.zeros(imgs.size(0), NUM_CLASSES, device=device)
            for m in models_list:
                for _ in range(n_tta):
                    aug    = torch.stack([tta_tf(i) for i in imgs])
                    probs += F.softmax(m(aug), dim=1)
            probs /= (len(models_list) * n_tta)
            y_true.extend(lbls.numpy())
            all_probs.extend(probs.cpu().numpy())
    all_probs = np.array(all_probs)
    return np.array(y_true), all_probs.argmax(axis=1), all_probs

# Reload all 3 checkpoints cleanly before ensemble
model_A.load_state_dict(torch.load(BACKBONE_CFG['A']['ckpt'], weights_only=True))
model_B.load_state_dict(torch.load(BACKBONE_CFG['B']['ckpt'], weights_only=True))
model_C.load_state_dict(torch.load(BACKBONE_CFG['C']['ckpt'], weights_only=True))
ensemble_models = [model_A, model_B, model_C]

print('Running v7 Heterogeneous Ensemble TTA-45 (3 models x 15 passes)...')
y_ens, p_ens, probs_ens = ensemble_predict_tta(ensemble_models, val_loader, n_tta=15)

acc_ens = np.mean(y_ens == p_ens)
f1_ens  = f1_score(y_ens, p_ens, average='macro', zero_division=0)

print('\n' + '='*65)
print(f'  [v7 Heterogeneous Ensemble TTA-45]  Accuracy : {acc_ens*100:.2f}%')
print(f'  [v7 Heterogeneous Ensemble TTA-45]  Macro F1 : {f1_ens:.4f}')
print('='*65)
print(f'  [v6 Homogeneous Ensemble TTA-30]    Accuracy : (see v6 results)')
print(f'  [v5 TTA-10]                         Accuracy : 83.67%')
print(f'  [v4 baseline]                       Accuracy : 81.83%')
print('='*65)
print(f'  Target >95%: {"ACHIEVED" if acc_ens >= 0.95 else f"Current: {acc_ens*100:.2f}%"}')


In [ ]:
cm_ens = confusion_matrix(y_ens, p_ens)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_ens, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(
    f'v7 Heterogeneous Ensemble TTA-45  (Acc={acc_ens*100:.2f}%)',
    fontsize=13
)
plt.tight_layout()
plt.savefig('cm_v7_ensemble.png', dpi=300, bbox_inches='tight')
plt.show()

mel_fp_ens_v7 = cm_ens[NV_IDX, MEL_IDX]
print(f'nv->mel false positives : {mel_fp_ens_v7}')
print(f'  (v5 had 182, v7 target: <30)')


In [ ]:
y_bin_ens = label_binarize(y_ens, classes=list(range(NUM_CLASSES)))
plt.figure(figsize=(10, 8))
colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2']
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin_ens[:, i], probs_ens[:, i])
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'{CLASS_NAMES[i]} AUC={auc(fpr,tpr):.3f}')
plt.plot([0,1],[0,1],'k--',lw=1)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — v7 Heterogeneous Ensemble TTA-45', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_v7_ensemble.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
def plot_clf_heatmap(y_true, y_pred, class_names,
                     title='ConvKAN-Derm v7',
                     save_path='clf_report.png',
                     cmap_name='YlGn', dpi=300):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(class_names))), zero_division=0)
    total  = support.sum()
    macro  = [precision.mean(), recall.mean(), f1.mean()]
    w_avg  = [(precision*support).sum()/total,
              (recall*support).sum()/total,
              (f1*support).sum()/total]
    class_data   = np.column_stack([precision, recall, f1])
    summary_data = np.array([macro, w_avg])
    cmap  = plt.cm.get_cmap(cmap_name)
    n_cls = len(class_names)
    fig   = plt.figure(figsize=(9, n_cls*0.68+3.6), facecolor='white')
    gs    = gridspec.GridSpec(3, 1, height_ratios=[n_cls, 0.4, 2], hspace=0.07)
    ax1, ax2 = fig.add_subplot(gs[0]), fig.add_subplot(gs[2])

    def _panel(ax, data, rows, show_x=False):
        im = ax.imshow(data, cmap=cmap, vmin=0, vmax=1, aspect='auto')
        for r in range(data.shape[0]):
            for c in range(data.shape[1]):
                v = data[r, c]
                ax.text(c, r, f'{v:.2f}', ha='center', va='center',
                        fontsize=13, fontweight='bold',
                        color='white' if v > 0.52 else '#1a1a1a')
        for x in np.arange(-0.5, 3, 1): ax.axvline(x, color='white', lw=2.5)
        for y in np.arange(-0.5, data.shape[0], 1): ax.axhline(y, color='white', lw=2)
        ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=12)
        ax.tick_params(left=False, bottom=False)
        ax.set_xticks(range(3))
        if show_x:
            ax.set_xticklabels(['Precision','Recall','F1-Score'],
                               fontsize=13, fontweight='bold')
            ax.set_xlabel('Evaluation Metrics', fontsize=13,
                          fontweight='bold', labelpad=10)
        else:
            ax.set_xticklabels([])
        for sp in ax.spines.values(): sp.set_visible(False)
        return im

    im1 = _panel(ax1, class_data,   class_names, show_x=False)
    _   = _panel(ax2, summary_data, ['macro avg','weighted avg'], show_x=True)
    ax1.set_ylabel('Target Pathology Classes', fontsize=13,
                   fontweight='bold', labelpad=12)
    fig.suptitle(title, fontsize=15, fontweight='bold',
                 color='#1b4332', y=0.99, va='top')
    cbar = fig.colorbar(im1, ax=[ax1,ax2], location='right',
                        fraction=0.028, pad=0.025, shrink=0.95)
    cbar.set_label('Metric Performance Scale', fontsize=11,
                   rotation=270, labelpad=16, color='#333333')
    cbar.ax.tick_params(labelsize=10); cbar.outline.set_visible(False)
    plt.savefig(save_path, dpi=dpi, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    print(f'Saved -> {save_path}  ({dpi} DPI)')
    plt.show(); plt.close()

print(classification_report(y_ens, p_ens, target_names=CLASS_NAMES, zero_division=0))
plot_clf_heatmap(
    y_ens, p_ens, CLASS_NAMES,
    title='ConvKAN-Derm v7 — Heterogeneous Ensemble Performance Profile\n'
          '(EffNetV2-L + ConvNeXt-B + Swin-B  |  HAM10000  |  TTA-45)',
    save_path='clf_report_v7_ensemble.png',
    cmap_name='YlGn', dpi=300
)


In [ ]:
def plot_model_history(hists_by_stage, model_label, save_name):
    all_loss = []; all_f1 = []; all_acc = []; stage_ends = []; stage_names = []
    stage_labels = ['S2','S3','S4']
    for i, (name, h) in enumerate(hists_by_stage):
        all_loss.extend(h['loss'])
        all_f1.extend(h['val_f1'])
        all_acc.extend(h['val_acc'])
        if i < len(hists_by_stage) - 1:
            stage_ends.append(len(all_loss))
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'ConvKAN-Derm v7 — {model_label} Training', fontsize=14, fontweight='bold')
    for ax, key, vals, ylabel, color in zip(
        axes,
        ['loss','val_f1','val_acc'],
        [all_loss, all_f1, all_acc],
        ['Loss','Macro F1','Accuracy'],
        ['steelblue','darkorange','seagreen']
    ):
        ax.plot(vals, color=color, lw=2)
        for se, sl in zip(stage_ends, stage_labels):
            ax.axvline(se-0.5, color='gray', ls='--', lw=1, alpha=0.7, label=sl)
        ax.set(xlabel='Epoch', ylabel=ylabel, title=ylabel)
        ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()

plot_model_history(
    [('S1',hist_A1),('S2',hist_A2),('S3',hist_A3),('S4',hist_A4)],
    'Model A (EfficientNetV2-L)', 'training_curves_v7_A.png'
)
plot_model_history(
    [('S1',hist_B1),('S2',hist_B2),('S3',hist_B3),('S4',hist_B4)],
    'Model B (ConvNeXt-Base)', 'training_curves_v7_B.png'
)
plot_model_history(
    [('S1',hist_C1),('S2',hist_C2),('S3',hist_C3),('S4',hist_C4)],
    'Model C (Swin-Base 384)', 'training_curves_v7_C.png'
)


### 21. ConvKAN Interpretability Head — v7 Upgrade

**v6 KAN head**: 256-d features (single EffNetV2-M backbone) → KAN(256, 7)
**v7 KAN head**: 768-d features (3 × 256-d concatenated) → KAN(768, 7)

The v7 KAN head is trained on the **frozen ensemble bottleneck**.
This means the B-spline visualisations now capture the **joint decision boundary**
across all three architectural perspectives simultaneously — a significant
upgrade in interpretability depth for the research paper.

> `best_v7_kan.pth` is a separate file. The three backbone checkpoints are NOT modified.


In [ ]:
class EnsembleKANHead_v7(nn.Module):
    def __init__(self, models_list, num_classes=NUM_CLASSES):
        super().__init__()
        self.models = nn.ModuleList(models_list)
        for m in self.models:
            for p in m.parameters():
                p.requires_grad = False     # freeze all 3 backbones
        # 3 models x 256-dim = 768-dim KAN input
        self.kan = KAN([768, num_classes])

    def forward(self, x):
        # Concatenate 256-d features from each frozen backbone
        feats = torch.cat([m.get_features(x) for m in self.models], dim=1)  # [B, 768]
        return self.kan(feats)

# Reload best checkpoints
model_A.load_state_dict(torch.load(BACKBONE_CFG['A']['ckpt'], weights_only=True))
model_B.load_state_dict(torch.load(BACKBONE_CFG['B']['ckpt'], weights_only=True))
model_C.load_state_dict(torch.load(BACKBONE_CFG['C']['ckpt'], weights_only=True))

kan_model_v7 = EnsembleKANHead_v7([model_A, model_B, model_C]).to(device)

kan_opt = torch.optim.AdamW(kan_model_v7.kan.parameters(), lr=5e-4, weight_decay=1e-3)
kan_sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    kan_opt, T_0=5, T_mult=2, eta_min=1e-6
)

print('Training v7 KAN head (768->7) on frozen ensemble bottleneck...')
print('  Input: [EffNetV2-L:256] ++ [ConvNeXt-B:256] ++ [Swin-B:256] = 768-d')
print('  KAN checkpoints -> best_v7_kan.pth (does NOT modify backbone .pth files)')

hist_kan_v7 = run_stage(
    kan_model_v7, train_loader_A, val_loader,
    kan_opt, criterion, kan_sch,
    num_epochs=15, stage_name='KAN-v7',
    checkpoint_path='best_v7_kan.pth',
    patience=10, use_aug=False, accum_steps=1
)


In [ ]:
def plot_kan_splines_v7(kan_model, num_splines=7, save_path='kan_splines_v7.png'):
    try:
        layer = kan_model.kan.layers[0]
    except Exception as e:
        print(f'Layer access error: {e}'); return
    attr = next((a for a in ['spline_weight','weight','coef']
                 if hasattr(layer, a)), None)
    if attr is None:
        print('Available attrs:', [a for a in dir(layer) if not a.startswith('_')]); return
    weights = getattr(layer, attr)
    n       = min(num_splines, weights.shape[1])
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3.5))
    if n == 1: axes = [axes]
    for i in range(n):
        w = weights[0, i].detach().cpu().numpy()
        axes[i].plot(w, lw=2, color=f'C{i}')
        axes[i].fill_between(range(len(w)), w, alpha=0.15, color=f'C{i}')
        axes[i].set_title(f'{CLASS_NAMES[i]}\n(B-spline)', fontsize=9)
        axes[i].grid(alpha=0.3)
        axes[i].set_xlabel('Knot index', fontsize=7)
    plt.suptitle(
        'ConvKAN-Derm v7 — Learnable B-Spline Decision Functions\n'
        'KAN Head: 768-d Ensemble Bottleneck → 7 Classes',
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Saved -> {save_path}')

# Session-interrupt guard
try:
    _ = kan_model_v7.kan
except NameError:
    print('Reconstructing EnsembleKANHead_v7 from checkpoints...')
    model_A.load_state_dict(torch.load(BACKBONE_CFG['A']['ckpt'], weights_only=True))
    model_B.load_state_dict(torch.load(BACKBONE_CFG['B']['ckpt'], weights_only=True))
    model_C.load_state_dict(torch.load(BACKBONE_CFG['C']['ckpt'], weights_only=True))
    kan_model_v7 = EnsembleKANHead_v7([model_A, model_B, model_C]).to(device)

kan_model_v7.load_state_dict(torch.load('best_v7_kan.pth', weights_only=True))
print('Loaded best_v7_kan.pth')
plot_kan_splines_v7(kan_model_v7)


### 23. ISIC 2019 Cross-Dataset Validation — The Overfitting Litmus Test

**Why this matters for publication**:
A >95% accuracy on HAM10000 alone is a **red flag** — reviewers will immediately
question whether the model memorised the dataset.

This cell freezes the v7 ensemble **as-is** (no retraining, no weight modification)
and evaluates it on the completely independent **ISIC 2019 dataset** (~25K images).

| Expected Outcome | Interpretation |
|---|---|
| ISIC-2019 >90% | Genuine feature learning. Model is publication-ready. |
| ISIC-2019 80-90% | Partial generalisation. Investigate per-class gaps. |
| ISIC-2019 <80% | Overfit alert — revisit regularisation (weight_decay, aug). |

> **Dataset isolation rule**: ISIC 2019 was never seen during training. The weights are frozen before this cell runs.


In [ ]:
import kagglehub as kh
try:
    path_isic = kh.dataset_download('andrewmvd/isic-2019')
    print(f'ISIC-2019 path: {path_isic}')
except Exception as e:
    print(f'Download error: {e}')

ISIC_TO_HAM = {
    'AK': 'akiec', 'BCC': 'bcc', 'BKL': 'bkl', 'DF': 'df',
    'MEL': 'mel',  'NV': 'nv',  'VASC': 'vasc', 'SCC': None
}

meta_files = glob.glob(os.path.join(path_isic, '**', '*.csv'), recursive=True)
gt_file    = next((f for f in meta_files if 'GroundTruth' in f), meta_files[0])
isic_meta  = pd.read_csv(gt_file)

class_cols = [c for c in ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
              if c in isic_meta.columns]
isic_meta['isic_class'] = isic_meta[class_cols].idxmax(axis=1)
isic_meta['dx']         = isic_meta['isic_class'].map(ISIC_TO_HAM)
isic_meta = isic_meta[isic_meta['dx'].notna()].reset_index(drop=True)
isic_meta['label_idx']  = isic_meta['dx'].map(label_map)

isic_img_dict = {
    os.path.splitext(os.path.basename(x))[0]: x
    for x in glob.glob(os.path.join(path_isic, '**', '*.jpg'), recursive=True)
}
img_col = 'image' if 'image' in isic_meta.columns else isic_meta.columns[0]
isic_meta['path'] = isic_meta[img_col].map(isic_img_dict)
isic_meta = isic_meta.dropna(subset=['path']).reset_index(drop=True)
print(f'ISIC-2019 usable samples (SCC excluded): {len(isic_meta)}')
print(isic_meta['dx'].value_counts())

isic_loader = DataLoader(
    HAM10000Dataset(isic_meta, val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

# Reload frozen ensemble — NO retraining
model_A.load_state_dict(torch.load(BACKBONE_CFG['A']['ckpt'], weights_only=True))
model_B.load_state_dict(torch.load(BACKBONE_CFG['B']['ckpt'], weights_only=True))
model_C.load_state_dict(torch.load(BACKBONE_CFG['C']['ckpt'], weights_only=True))

print('\nEvaluating v7 ensemble on ISIC-2019 (FROZEN weights, zero-shot)...')
yi_t, yi_p, yi_probs = ensemble_predict_tta(
    [model_A, model_B, model_C], isic_loader, n_tta=15
)

isic_acc_v7 = np.mean(yi_t == yi_p)
isic_f1_v7  = f1_score(yi_t, yi_p, average='macro', zero_division=0)
print(f'\nISIC-2019 Ensemble Accuracy : {isic_acc_v7*100:.2f}%')
print(f'ISIC-2019 Ensemble Macro F1 : {isic_f7:.4f}')

plot_clf_heatmap(
    yi_t, yi_p, CLASS_NAMES,
    title='ConvKAN-Derm v7 — ISIC-2019 Cross-Validation Profile\n'
          '(Zero-shot transfer | Heterogeneous Ensemble | TTA-45)',
    save_path='clf_report_v7_isic2019.png',
    cmap_name='YlOrRd', dpi=300
)


In [ ]:
# ── Frozen reference results from earlier versions ────────────────────────
V4_ACC, V4_F1 = 81.83, 0.8066
V5_ACC, V5_F1 = 83.67, 0.8268
# v6 results: fill in from UROP_v6.ipynb final cell output
V6_HAM_ACC  = float(input('Enter v6 HAM10000 Ensemble Accuracy (e.g. 91.23): ') or '0.0')
V6_HAM_F1   = float(input('Enter v6 HAM10000 Ensemble Macro F1 (e.g. 0.9012): ') or '0.0')
V6_ISIC_ACC = float(input('Enter v6 ISIC-2019 Accuracy (e.g. 78.45): ') or '0.0')
V6_ISIC_F1  = float(input('Enter v6 ISIC-2019 Macro F1 (e.g. 0.7500): ') or '0.0')

v7_ham_acc  = acc_ens * 100
v7_ham_f1   = f1_ens
v7_isic_acc = isic_acc_v7 * 100
v7_isic_f1  = isic_f1_v7

print('\n' + '='*85)
print(f'  {"Version + Config":<32} {"HAM Acc":>9} {"HAM F1":>8} {"ISIC Acc":>10} {"ISIC F1":>8}  {"Role"}')
print('-'*85)
print(f'  {"v4  ResNet50 224x224":<32} {V4_ACC:>8.2f}% {V4_F1:>8.4f}  {"—":>9}   {"—":>7}  Frozen baseline')
print(f'  {"v5  EffNetV2-S TTA-10":<32} {V5_ACC:>8.2f}% {V5_F1:>8.4f}  {"—":>9}   {"—":>7}  Frozen baseline')
print(f'  {"v6  Homogeneous Ens TTA-30":<32} {V6_HAM_ACC:>8.2f}% {V6_HAM_F1:>8.4f}  '
      f'{V6_ISIC_ACC:>8.2f}%  {V6_ISIC_F1:>7.4f}  HAM master')
print(f'  {"v7  Heterogeneous Ens TTA-45":<32} {v7_ham_acc:>8.2f}% {v7_ham_f1:>8.4f}  '
      f'{v7_isic_acc:>8.2f}%  {v7_isic_f1:>7.4f}  HAM master v7')
print('='*85)
print(f'\n  Accuracy gain  v4 → v7 : {v7_ham_acc - V4_ACC:+.2f} pp')
print(f'  Accuracy gain  v5 → v7 : {v7_ham_acc - V5_ACC:+.2f} pp')
print(f'  Accuracy gain  v6 → v7 : {v7_ham_acc - V6_HAM_ACC:+.2f} pp')
print(f'  Generalisation gap (HAM → ISIC) v7: {v7_ham_acc - v7_isic_acc:+.2f} pp')
print(f'\n  >95%% target: {"ACHIEVED ✓" if v7_ham_acc >= 95.0 else f"Current best: {v7_ham_acc:.2f}%%"}')
print(f'  ISIC >90%%  : {"ACHIEVED ✓" if v7_isic_acc >= 90.0 else f"Current: {v7_isic_acc:.2f}%%"}')
print('\n  Paper figures saved at 300 DPI:')
for f in [
    'cm_v7_ensemble.png',
    'roc_v7_ensemble.png',
    'clf_report_v7_ensemble.png   <- primary paper figure',
    'clf_report_v7_isic2019.png',
    'kan_splines_v7.png',
    'training_curves_v7_A.png',
    'training_curves_v7_B.png',
    'training_curves_v7_C.png',
]:
    print(f'    {f}')


### 25. Checkpoint Registry — v7 Files

| Checkpoint | Description | Do Not Delete |
|---|---|---|
| `best_v7_A_effv2l.pth` | EfficientNetV2-L best F1 weights | ✓ |
| `best_v7_B_convnext.pth` | ConvNeXt-Base best F1 weights | ✓ |
| `best_v7_C_swinb.pth` | Swin-Base 384 best F1 weights | ✓ |
| `best_v7_kan.pth` | KAN head (768→7, frozen ensemble bottleneck) | ✓ |

**v6 checkpoints are preserved and untouched:**
`best_v6_m.pth`, `best_v6_ens_S_123.pth`, `best_v6_ens_S_456.pth`, `best_v6_kan.pth`

If v7 fails to beat v6, the v6 weights remain the primary paper result.
All subsequent experiments must be versioned as v8+.
